# CNN–BiLSTM–Attention : apprentissage avec reprise dans Colab

Entraînement sur les vrais **CSV NASA prétraités** versionnés dans ce dépôt. Ce nouveau pipeline reprend l'architecture des notebooks historiques, mais les entrées diffèrent des traces MATLAB brutes : **ne pas considérer les scores comme une reproduction directe de l'article**.

B0018 : entraînement B0005 + B0006, validation B0007, test indépendant B0018. Le SOH cible est calculé à partir de la capacité de chaque batterie, mais cette capacité **n'entre pas** dans le réseau. Le scaler est ajusté uniquement sur les deux batteries d'entraînement.

Les snapshots Keras complets sont sauvegardés après chaque epoch dans Google Drive. La reprise restaure aussi l'état de l'optimiseur Adam.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPOSITORY = "Attention-Enhanced-CNN-BiLSTM-Framework-for-Lithium-Ion-Battery-State-of-Health-Prediction"
GITHUB_REPO = f"walidmchara/{REPOSITORY}"
repo = Path("/content") / REPOSITORY

if not repo.exists():
    subprocess.run(["git", "clone", f"https://github.com/{GITHUB_REPO}.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(repo)
print("GitHub clone :", repo)
print("Commit :", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import importlib.util
for module, package in [("numpy","numpy"),("pandas","pandas"),("sklearn","scikit-learn"),("tensorflow","tensorflow"),("matplotlib","matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import tensorflow as tf
print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

EXPERIMENT = "B0018"  # Après cette expérience : "B0030"
EPOCHS = 30
BATCH_SIZE = 16
SEED = 42
LEARNING_RATE = 0.001
RUN_ID = f"{EXPERIMENT.lower()}_e{EPOCHS}_seed{SEED}"

persistent_run = Path("/content/drive/MyDrive/NASA_CNN_BiLSTM_Attention") / RUN_ID
checkpoint_dir = persistent_run / "checkpoints"
output_dir = persistent_run / "outputs"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Dataset :", repo / "data/processed/nasa")
print("Dossier permanent :", persistent_run)
print("Reprise disponible :", (checkpoint_dir / "latest.json").exists())

In [ ]:
import sys
sys.path.insert(0, str(repo))
from scripts.train_resumable import prepare_data
(train_x, train_y), (val_x, val_y), (test_x, test_y), provenance, _, _ = prepare_data(
    repo / "data/processed/nasa", EXPERIMENT
)
print("Train :", train_x.shape, "Validation :", val_x.shape, "Test :", test_x.shape)
print("Cellules :", provenance["split"])
print("Colonnes d'entrée :", provenance["features"])

## Démarrer ou reprendre 30 epochs

Le dossier de sauvegarde est stable. Si Colab interrompt la session, exécute de nouveau les mêmes cellules. Le script détecte le dernier epoch entièrement sauvegardé et reprend au suivant. **Ne change pas les hyperparamètres, la seed, les données ou la durée totale de cette expérience en cours de reprise.**

In [ ]:
command = [
    sys.executable, "-u", "scripts/train_resumable.py",
    "--experiment", EXPERIMENT, "--data-dir", "data/processed/nasa",
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE), "--seed", str(SEED),
    "--checkpoint-dir", str(checkpoint_dir), "--output-dir", str(output_dir),
]
if (checkpoint_dir / "latest.json").exists():
    command.append("--resume")
print("Training :", RUN_ID, flush=True)
subprocess.run(command, cwd=repo, check=True)
print("Entraînement terminé")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
metrics = json.loads((output_dir / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics["test"], indent=2))
print("Meilleur epoch :", metrics["best_validation_epoch"])
prediction = pd.read_csv(output_dir / "test_predictions.csv")
history = pd.read_csv(output_dir / "training_history.csv")
plt.figure(figsize=(9, 3))
plt.plot(prediction["cycle"], prediction["soh_true"], label="SOH mesuré")
plt.plot(prediction["cycle"], prediction["soh_pred"], label="SOH estimé")
plt.xlabel("Cycle"); plt.ylabel("SOH"); plt.title(f"Test indépendant {EXPERIMENT}")
plt.legend(); plt.tight_layout(); plt.show()
plt.figure(figsize=(9, 3))
plt.plot(history["epoch"], history["loss"], label="MSE entraînement")
plt.plot(history["epoch"], history["val_loss"], label="MSE validation")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.tight_layout(); plt.show()

## Publier les petits résultats dans GitHub

Facultatif, après la fin réussie de l'entraînement. Le secret Colab **GithubToken** doit autoriser **Contents: Read and write** pour CE dépôt public. Les résultats publiés incluent métriques, historique, paramètres, scaler et prédictions. Les checkpoints Keras et les archives brutes restent sur Google Drive, et ne sont pas ajoutés au dépôt.

In [ ]:
from google.colab import userdata
token = userdata.get("GithubToken")
if not token:
    raise RuntimeError("Configurer GithubToken dans les Secrets Colab avec écriture autorisée sur ce dépôt")

if (repo / "results" / RUN_ID / "manifest.json").exists():
    print("Résultats déjà présents dans ce clone :", RUN_ID)
else:
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    subprocess.run(
        [sys.executable, "scripts/publish_results.py",
         "--source", str(output_dir), "--run-id", RUN_ID],
        cwd=repo, env=env, check=True,
    )
print(f"GitHub : https://github.com/{GITHUB_REPO}/tree/main/results/{RUN_ID}")